In [1]:
import argparse
import os
import random,numpy,pandas
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

In [2]:
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data
import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.utils as vutils
import torchvision.transforms.functional as RF
from PIL import Image
import numpy as np
import random,cv2,pandas,os,numpy
import matplotlib.pyplot as plt
import matplotlib.animation as animation

In [3]:
seed = 999
print("Random Seed: ", seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

nz = 100
beta1 = 0.5
lr = 0.0001
batch_size=100000
ngpu=1
ngf,nc = 3,3
ndf = 64

device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

Random Seed:  999


In [4]:
x = pandas.read_csv("/kaggle/input/rohlik-sales-forecasting-challenge-v2/sales_train.csv").sort_values(by=['date', 'warehouse'])
x = torch.from_numpy(numpy.nan_to_num(x[['total_orders', 'sell_price_main', 'type_0_discount', 'type_1_discount',
                                         'type_2_discount', 'type_3_discount', 'type_4_discount',  'type_5_discount', 
                                         'type_6_discount']].to_numpy().astype(numpy.float32)).reshape((13, 308263, 9)))
x.shape #4315682

torch.Size([13, 308263, 9])

In [5]:
class RF_NET(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.rafire = nn.Sequential(       
            nn.Linear(9, 1524),
            nn.Linear(1524, 824),
            nn.Linear(824, 424),
            nn.Linear(424, 124),
            nn.Linear(124, 1)
        )

    def forward(self,x):
        
        return self.rafire(x)


In [6]:
rf_net = RF_NET().type(torch.float32).to(device).eval()
rf_net= nn.DataParallel(rf_net).to(device)

rf_net.load_state_dict(torch.load("/kaggle/input/encoder-weight-data/weight.pth",weights_only=False,map_location=torch.device('cpu')))

<All keys matched successfully>

In [7]:
j=0
for i in x:
    print(i.shape)
    if j==0:
        encode = rf_net(i).cpu().detach().numpy()
    else:
        encode = numpy.concatenate((encode, rf_net(i).cpu().detach().numpy()), axis=0)
    print(encode.shape)
    #if j==25:
    #    break
    j+=1
#encode = encode.reshape(-1)

torch.Size([308263, 9])


/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):


(308263, 1)
torch.Size([308263, 9])
(616526, 1)
torch.Size([308263, 9])


/opt/conda/lib/python3.10/site-packages/torch/nn/modules/linear.py:117: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /usr/local/src/pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:135.)
  return F.linear(input, self.weight, self.bias)


(924789, 1)
torch.Size([308263, 9])
(1233052, 1)
torch.Size([308263, 9])
(1541315, 1)
torch.Size([308263, 9])
(1849578, 1)
torch.Size([308263, 9])
(2157841, 1)
torch.Size([308263, 9])
(2466104, 1)
torch.Size([308263, 9])
(2774367, 1)
torch.Size([308263, 9])
(3082630, 1)
torch.Size([308263, 9])
(3390893, 1)
torch.Size([308263, 9])
(3699156, 1)
torch.Size([308263, 9])
(4007419, 1)


In [8]:
encode.shape

(4007419, 1)

In [9]:
torch.save(torch.Tensor(encode.reshape(-1)).reshape((len(encode),1)), 'train_tensor.pt')
torch.Tensor(encode.reshape(-1).reshape((len(encode),1))).shape

torch.Size([4007419, 1])